# 📄 RAG Pipeline with LangChain, FAISS & HuggingFace

This notebook implements a complete **Retrieval-Augmented Generation (RAG)** pipeline from scratch. It is organized into three stages:

| Stage | File | Purpose |
|-------|------|---------|
| 1️⃣ Ingestion | `ingest.py` | Load PDF → Split → Embed → Store in FAISS |
| 2️⃣ Query | `query.py` | Load FAISS index → Semantic similarity search |
| 3️⃣ RAG Generation | `rag.py` | Retrieve context → Feed to LLM → Generate answer |

**Tech Stack:** `LangChain` · `FAISS` · `HuggingFace Transformers` · `sentence-transformers` · `Flan-T5`

---


## 🔧 Step 0 — Install Dependencies

Install all required libraries before running the pipeline.


In [1]:
!pip install langchain langchain-community faiss-cpu sentence-transformers transformers pypdf -q


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


---

## 📥 Stage 1 — Document Ingestion (`ingest.py`)

### What happens here?
1. **Load** the PDF using `PyPDFLoader` — converts each page into a LangChain `Document` object.
2. **Split** the text into smaller overlapping chunks using `RecursiveCharacterTextSplitter`.
   - `chunk_size=500` — each chunk holds ~500 characters
   - `chunk_overlap=50` — 50-character overlap ensures context isn't lost at boundaries
3. **Embed** each chunk using the `all-MiniLM-L6-v2` sentence-transformer model — converts text → 384-dimensional vectors.
4. **Store** all vectors in a **FAISS** index and save it locally for reuse.

> 💡 FAISS (Facebook AI Similarity Search) enables blazing-fast approximate nearest-neighbour search over millions of vectors.


In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters.character import RecursiveCharacterTextSplitter

# ── Load PDF ──────────────────────────────────────────────────
# Replace with your own PDF file path
loader = PyPDFLoader("clustering.pdf")
docs = loader.load()

print(f"Total pages loaded: {len(docs)}")
print(f"\nSample content (first 300 chars of page 1):\n{docs[0].page_content[:300]}")



Total pages loaded: 16

Sample content (first 300 chars of page 1):
Clustering Algorithms
1 Introduction to Clustering
Clustering is anunsupervised learningtechnique that aims to group a set of data
objects into clusters such that objects within the same cluster are more similar to each
other than to those in other clusters. Similarity is usually measured using dist


In [3]:
# ── Chunk the documents ───────────────────────────────────────
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(docs)

print(f"Total chunks created: {len(chunks)}")
print(f"\nSample chunk:\n{chunks[0].page_content}")


Total chunks created: 72

Sample chunk:
Clustering Algorithms
1 Introduction to Clustering
Clustering is anunsupervised learningtechnique that aims to group a set of data
objects into clusters such that objects within the same cluster are more similar to each
other than to those in other clusters. Similarity is usually measured using distance metrics
such as Euclidean, Manhattan, or cosine distance.
1.1 Objectives of Clustering
Clustering aims to organize unlabeled data into meaningful groups based on similarity


In [4]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# ── Embed & index ─────────────────────────────────────────────
# Downloads ~90MB model on first run
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

db = FAISS.from_documents(chunks, embedding)

# ── Save FAISS index to disk ──────────────────────────────────
db.save_local("faiss_index")

print("✅ FAISS index saved to ./faiss_index/")


C:\Users\KAUSHIK\AppData\Local\Temp\ipykernel_7072\1461050294.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(


✅ FAISS index saved to ./faiss_index/


---

## 🔍 Stage 2 — Semantic Search / Query (`query.py`)

### What happens here?
1. **Load** the saved FAISS index back into memory.
2. Accept a **natural language query** from the user.
3. Perform **similarity search** — embeds the query, finds the top-k closest chunks in vector space.
4. Return and display the most relevant document chunks.

> 💡 `k=3` means we retrieve the 3 most semantically similar chunks to the query.


In [5]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

# ── Reload the FAISS index ────────────────────────────────────
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

db = FAISS.load_local(
    "faiss_index",
    embedding,
    allow_dangerous_deserialization=True   # required for local FAISS indexes
)

print("✅ FAISS index loaded successfully.")


✅ FAISS index loaded successfully.


In [6]:
# ── Run a similarity search ───────────────────────────────────
# Change this query to test different questions
query = "What is Hierarchical clustering"

docs = db.similarity_search(query, k=3)

for i, doc in enumerate(docs):
    print(f"\n{'='*50}")
    print(f"Result {i+1}:")
    print(f"{'='*50}")
    print(doc.page_content)



Result 1:
6 Hierarchical Clustering
6.1 Overview
Hierarchical clustering is an unsupervised clustering technique that organizes data into
a hierarchy of nested clusters. Unlike partition-based methods, hierarchical clustering
does not require the number of clusters to be specified in advance. Instead, it produces
a multilevel clustering structure that can be visualized using a tree-like representation
known as a dendrogram.
6.2 Types of Hierarchical Clustering

Result 2:
6.2 Types of Hierarchical Clustering
Hierarchical clustering algorithms are broadly classified based on the direction in which
the hierarchy is constructed.
•Agglomerative (bottom-up) clustering:Agglomerative clustering begins with
each data point treated as an individual cluster. At each iteration, the two closest
clusters are merged based on a predefined linkage criterion. This merging process
continues until all data points are combined into a single cluster. Agglomerative

Result 3:
clustering is the most commonly

---

## 🤖 Stage 3 — Full RAG Pipeline (`rag.py`)

### What happens here?
This is the complete **Retrieval-Augmented Generation** loop:

```
User Query
    │
    ▼
[FAISS Retriever] ── semantic search ──► Top-k Chunks (Context)
    │
    ▼
[Prompt Builder] ── wraps query + context into structured prompt
    │
    ▼
[Flan-T5 LLM] ── generates grounded answer from context only
    │
    ▼
Answer printed to user
```

**Key design decisions:**
- **Score filtering** (`score < 1.5`) — only includes chunks that are genuinely relevant; falls back to top-2 if nothing passes.
- **Context truncation** (`[:1000]`) — prevents token overflow in the LLM.
- **Structured prompt** — instructs the LLM to answer *only* from context, avoiding hallucination.
- **Flan-T5-base** — a free, open-source text2text model by Google; no API key needed.


In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline

# ── Load embeddings + FAISS index ─────────────────────────────
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

db = FAISS.load_local(
    "faiss_index",
    embedding,
    allow_dangerous_deserialization=True
)

print("✅ Embeddings and FAISS index ready.")


✅ Embeddings and FAISS index ready.


: 

In [ ]:
# ── Load LLM (Flan-T5-base — Free & Local) ────────────────────
# First run will download ~990MB model weights
pipe = pipeline(
    "text2text-generation",
    model="google/flan-t5-large",
    max_new_tokens=256,
    temperature=0.3
)

llm = HuggingFacePipeline(pipeline=pipe)

print("✅ LLM loaded: google/flan-t5-base")


In [ ]:
# ── Retrieval Function with Score Filtering ───────────────────
def retrieve_docs(query):
    """
    Retrieve the most relevant chunks for a query.
    - Uses FAISS similarity_search_with_score (L2 distance)
    - Filters out chunks with score >= 1.5 (too dissimilar)
    - Falls back to top-2 chunks if all are filtered out
    """
    docs_with_scores = db.similarity_search_with_score(query, k=3)

    # Keep only high-relevance chunks (lower L2 score = more similar)
    filtered_docs = [doc for doc, score in docs_with_scores if score < 1.5]

    # Fallback: if nothing passes the threshold, use top-2 anyway
    if not filtered_docs:
        filtered_docs = [doc for doc, _ in docs_with_scores[:2]]

    return filtered_docs[:2]

print("✅ retrieve_docs() function defined.")


✅ retrieve_docs() function defined.


In [ ]:
# ── Prompt Builder ─────────────────────────────────────────────
def build_prompt(query, context):
    """
    Wraps the user query and retrieved context into a structured prompt
    that instructs the LLM to answer ONLY from the provided context.
    """
    return f"""
You are an AI tutor.

Instructions:
- Answer ONLY using the context below
- If the answer is not in the context, say: Not in document
- Be clear and structured
- Do not add extra knowledge

Context:
{context}

Question:
{query}

Answer in this format:
- Explanation:
- Key Points:
"""


In [ ]:
# ── Single Query (Notebook-friendly, no while loop) ───────────
# Change this to test different questions
query = "What is multi-head attention?"

# Step 1: Retrieve relevant chunks
docs = retrieve_docs(query)

# Step 2: Show retrieved context
print("--- Retrieved Context ---")
for i, doc in enumerate(docs):
    print(f"\nChunk {i+1}:\n{doc.page_content[:200]}")

# Step 3: Build context string
context = "\n\n".join([doc.page_content for doc in docs])
context = context[:1000]   # truncate to prevent token overflow

# Step 4: Build prompt
prompt = build_prompt(query, context)

# Step 5: Generate answer
response = llm.invoke(prompt)

print("\n" + "="*50)
print("Answer:")
print("="*50)
print(response)


--- Retrieved Context ---

Chunk 1:
quiring external supervision. This helps in understanding the internal organization
of data and provides insights into the domain being analyzed.
•Serve as a preprocessing step for other learning task

Chunk 2:
a centroid or medoid, allowing analysts to summarize and interpret massive datasets
more efficiently. This abstraction is especially useful for visualization, reporting,
and decision-making.
•Identify

Answer:
Not in document


---

## 🔁 Optional — Interactive Q&A Loop

Run the cell below to enter an interactive question-answering session. Type `exit` to quit.

> ⚠️ **Note:** `input()` works in Jupyter but may not work in all notebook environments (e.g., JupyterLite). Use the single-query cell above for guaranteed compatibility.


In [ ]:
# ── Interactive Q&A loop ──────────────────────────────────────
while True:
    query = input("\nAsk (or type 'exit' to quit): ").strip()

    if query.lower() == "exit":
        print("Goodbye!")
        break

    docs = retrieve_docs(query)

    print("\n--- Retrieved Context ---")
    for i, doc in enumerate(docs):
        print(f"\nChunk {i+1}:\n{doc.page_content[:200]}")

    context = "\n\n".join([doc.page_content for doc in docs])
    context = context[:1000]

    prompt = build_prompt(query, context)
    response = llm.invoke(prompt)

    print("\n" + "="*50)
    print("Answer:")
    print("="*50)
    print(response)



--- Retrieved Context ---

Chunk 1:
divide customers into distinct segments based on purchasing behavior, demograph-
ics, or preferences. This enables targeted marketing, personalized recommendation
systems, customer retention strategie

Chunk 2:
•Anomaly and fraud detection
•Image segmentation and pattern recognition
•Network intrusion detection
16

Answer:
Clustering is widely used to detect communities within social networks by grouping users with strong interac- tion or similarity patterns. This enables targeted marketing, personalized recommendation systems, customer retention strategies, and improved business decision-making. •Social network analysis and community detection: Clustering is widely used to detect communities within social networks by grouping users with strong interac- tion or similarity patterns. •Anomaly and fraud detection: Image segmentation and pattern recognition is widely used to detect communities within social networks by grouping users with strong interac

KeyboardInterrupt: 

---

## 📌 Summary & Key Concepts

| Concept | Details |
|---------|---------|
| **RAG** | Combines retrieval (FAISS) + generation (LLM) to ground answers in real documents |
| **Chunking** | Splits large docs into overlapping pieces to preserve context at boundaries |
| **Embeddings** | `all-MiniLM-L6-v2` maps text → 384-dim vectors capturing semantic meaning |
| **FAISS** | Efficient vector store for approximate nearest-neighbour search |
| **Score filtering** | L2 distance < 1.5 ensures only genuinely relevant chunks are used |
| **Flan-T5** | Free, open-source seq2seq LLM — no API key required |
| **Prompt engineering** | Structured prompt restricts LLM to context, reducing hallucination |

### 🚀 Possible Extensions
- Swap Flan-T5 for a larger model (e.g., `flan-t5-large`, Mistral via Ollama)
- Add a Streamlit / Gradio UI for a chatbot interface
- Support multiple PDFs by batching ingestion
- Use `ConversationalRetrievalChain` for multi-turn memory
